# Crypto oscillator finder — KRAKEN v9 (`recycle_v11` engine)

**v11.1** adds: a dust filter (`min_vol_usd`), quote-unit conversion for crypto-quoted
pairs (fund / min-order / volume-cap all in the pair's quote currency; exports carry both
`max_fund_value_quote` in quote units and `max_fund_value_usd`), finalize reusing the same
order-book snapshot the engine already sampled (no spurious spread gates), the
`rc_gate_two_sided_mode` holdout waiver, and a book-verification cell (Cell 10).

**What changed vs v10** (full details in `LadderLab_v11_notes_claude_code.md`):

1. **True holdout** — the deploy ladder is fitted *without* the last 15 days; the fitted,
   train-anchored ladder is scored once on that unseen tail. `holdout_edge_pct` is the only
   fully out-of-sample number for the geometry you deploy, and it gates deployment.
2. **Clean-block gating** — the 180d frozen report tags blocks that overlap the fit window
   (`in_fit`); the band-aware v10 gates must now pass on the *clean* blocks too.
3. **Fill realism** — fills require price to trade *through* the rung
   (`rc_fill_penetration_pct` + 1 tick), and per-bar filled notional is capped at
   `rc_volume_cap_frac` × the bar's quote volume, with partial fills (bars now carry volume).
4. **Slip symmetry** — the measured live order-book half-spread joins Roll + floor in the
   *search and WF*, not just as a finalize gate.
5. **Grid-harvest screen** — a model-free zig-zag swing count (net of round-trip cost) ranks
   the review set; it cannot overfit because nothing is fitted.
6. **Deep history** — MEXC-first Nx6 (with volume) history; Kraken USD pairs get months of hourly/5m depth via the price-guarded MEXC USDT-alias proxy (Kraken-native OHLC caps at 720 candles ≈ 30d of 1h, kept only as a fallback).

Nothing was removed from the v10 reports — v11 only adds columns/files. The
`*_copy_paste_ladders.md` is rendered by v10's renderer, byte-identical format.


In [1]:
# ── Cell 1: bootstrap ────────────────────────────────────────────────
# numba makes the engine ~50-100x faster; strongly recommended.
import sys, subprocess
for _mod, _pkg in (('numba', 'numba'), ('tabulate', 'tabulate')):
    try:
        __import__(_mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=False)

import importlib, os, time, json
from pathlib import Path
import numpy as np
import pandas as pd

import ladder_lab as ll
import ladder_lab_recycle as lr10          # v10 stays installed; v11 reuses it
import ladder_lab_recycle_v11 as lr
importlib.reload(ll); importlib.reload(lr10); importlib.reload(lr)

EXCHANGE = 'kraken'

print(f'ladder_lab {ll.__version__} | v10 {lr10.__version__} | v11 {lr.__version__} | numba={ll.HAVE_NUMBA}')
assert ll.parity_check(), 'base kernel parity FAILED'
assert lr10.recycle_parity_check(), 'v10 kernel parity FAILED'
assert lr.recycle_v11_parity_check(verbose=True), 'v11 kernel parity / v10-equivalence FAILED'
print('kernel parity OK (v11 reproduces v10 exactly with realism features off)')

CFG = lr.recycle_default_config(EXCHANGE)
# ---- knobs you will most likely touch ----
CFG['always_review_markets'] = ('XMR/USD', 'XMR/USDT', 'DASH/USD')
CFG['focus_markets']         = ()
CFG['rc_top_n_from_screen']  = 40      # markets from the screener that get the full engine
CFG['rc_n_candidates']       = 240     # per fit; drop to ~80 if running without numba
CFG['rc_target_trades_per_15d'] = 40   # recalibrate with the diagnostic-JSONL cell at the bottom
CFG['rc_report_interval']    = '5m'    # frozen reports / YAML eval / holdout on 5m bars
# ---- v11 realism knobs (defaults are the recommended settings) ----
CFG['rc_holdout_days']         = 15    # fit never sees the last N days (0 disables; don't)
CFG['rc_fill_penetration_pct'] = 0.0005  # price must trade 5bps+1tick THROUGH a rung to fill
CFG['rc_volume_cap_frac']      = 0.25  # max share of a bar's quote volume we may fill
CFG['rc_book_spread_in_slip']  = True  # live half-spread joins Roll+floor in search/WF too
# ---- v11.1 knobs ----
CFG['min_vol_usd']             = 10_000  # DUST FILTER: drop markets under this 24h USD volume
                                         # (the first run spent ~90% of engine time on books
                                         #  that could not absorb even the $200 fund floor)
CFG['rc_gate_two_sided_mode']  = 'either'  # 'blocks'=v10 | 'holdout' | 'either' (waive the
                                         # block two-sided gate when the HOLDOUT was two-sided;
                                         # re-anchored ladders are structurally one-sided vs a
                                         # trending past -- the holdout has no such bias)
# CFG['rc_v11_fill_model'] = False     # A/B: byte-identical v10 fills (leave True to deploy)

# Where your live controller YAMLs live (searched recursively):
YAML_ROOTS = ['.', 'controllers', str(Path.home())]
YAML_ROOTS = ['./controllers/kraken']

# artifacts/{exchange}/{timestamp}/files
RUN_TS = time.strftime('%Y%m%d-%H%M%S')
ART = Path('artifacts') / EXCHANGE / RUN_TS / 'files'
ART.mkdir(parents=True, exist_ok=True)
PREFIX = str(ART / f'KRAKEN_recycle_v9')
print('artifacts →', ART)


ladder_lab 1.0.0 | v10 10.2.0-recycle-rollspread-bandgates | v11 11.3.1-resilient-universe | numba=True
  series 0: v10-equiv=OK trades=20
  series 1: v10-equiv=OK trades=27
  series 2: v10-equiv=OK trades=18
  series 3: v10-equiv=OK trades=17
kernel parity OK (v11 reproduces v10 exactly with realism features off)
artifacts → artifacts/kraken/20260713-040411/files


In [2]:
# ── Cell 2: universe ─────────────────────────────────────────────────
# v11.1.3: build the universe WITHOUT the volume floor so live-YAML and
# always-review markets can never be starved of history (a live ZANO ladder
# was silently dropped when its 24h volume dipped under the floor). The dust
# filter is applied in Cell 4 to the SCREENED candidates only -- engine time
# is still protected, your own markets always get evaluated.
# v11.3.1: patient retries (15/30/45/60s -- outlasts a rate-limit window)
# and a loud fallback to the last good universe cached on disk (<=48h old),
# so a flaky /market/getlist costs freshness, not the whole run.
uni = lr.build_universe(EXCHANGE, CFG)
VOL_USD = dict(zip(uni['df'].pairkey, uni['df'].vol_usd))
cache = ll.CandleCache(CFG['cache_dir'], CFG['cache_ttl_hours'])


1513 kraken markets across 23 quote(s) >= $0 24h vol (USD-equiv).
  by quote: USD:691, EUR:584, USDT:49, USDC:47, XBT:32, GBP:27, ETH:19, AUD:14, CAD:11, JPY:10, CHF:6, USD1:3, EURC:3, SOL:3, FIDD:2, DAI:2, AUSD:2, PYUSD:2, EUROP:2, RLUSD:1, USDQ:1, USDD:1, USDR:1


In [3]:
# ── Cell 3: daily history + screener (breadth pass, unchanged from base) ──
hist_d = ll.prefetch_history(uni, CFG, cache)
screen_df = ll.screen(uni, hist_d, CFG)
screen_df = lr.drop_stable_stable(screen_df, CFG)
screen_df.head(25)


  ...MEXC pass 50/1513
  ...MEXC pass 100/1513
  ...MEXC pass 150/1513
  ...MEXC pass 200/1513
  ...MEXC pass 250/1513
  ...MEXC pass 300/1513
  ...MEXC pass 350/1513
  ...MEXC pass 400/1513
  ...MEXC pass 450/1513
  ...MEXC pass 500/1513
  ...MEXC pass 550/1513
  ...MEXC pass 600/1513
  ...MEXC pass 650/1513
  ...MEXC pass 700/1513
  ...MEXC pass 750/1513
  ...MEXC pass 800/1513
  ...MEXC pass 850/1513
  ...MEXC pass 900/1513
  ...MEXC pass 950/1513
  ...MEXC pass 1000/1513
  ...MEXC pass 1050/1513
  ...MEXC pass 1100/1513
  ...MEXC pass 1150/1513
  ...MEXC pass 1200/1513
  ...MEXC pass 1250/1513
  ...MEXC pass 1300/1513
  ...MEXC pass 1350/1513
  ...MEXC pass 1400/1513
  ...MEXC pass 1450/1513
  ...MEXC pass 1500/1513
  native fallback for 1412 markets (serial ~1/s) ...
  ...native 25/1412
  ...native 50/1412
  ...native 75/1412
  ...native 100/1412
  ...native 125/1412
  ...native 150/1412
  ...native 175/1412
  ...native 200/1412
  ...native 225/1412
  ...native 250/1412
  ...nativ

,days,price,low,high,ann_vol,er,cross_per_yr,in_band,net_return,cur_vs_high,...,coin,quote,src,vol_usd,min_qty,stale,tier,range_x,qualifies,composite
0,202,2.735000e-04,2.510750e-04,2.779000e-04,1.960575,0.002576,77.698020,0.648515,0.020522,0.526063,...,COPM,USD,Kraken,5.200648,1.800000e+04,False,full,1.11,True,0.995
1,220,6.340000e-01,6.445000e-01,9.256000e-01,1.527115,0.042687,77.977273,0.618182,-0.366000,0.604385,...,REP,EUR,Kraken,4.187925,5.000000e+00,False,full,1.44,True,0.979
2,220,7.890000e-01,7.355000e-01,1.137050e+00,1.569286,0.034396,97.886364,0.609091,-0.327366,0.665261,...,REP,USD,Kraken,21.667626,5.000000e+00,False,full,1.55,True,0.978
3,187,9.985000e-01,9.985000e-01,9.998000e-01,2.594340,0.000268,46.844920,0.973262,-0.001500,0.269866,...,FRNT,USD,Kraken,4.992500,5.000000e+00,False,full,1.00,True,0.977
4,158,2.709000e-03,2.506500e-03,7.299800e-03,3.535260,0.089982,62.373418,0.594937,-0.735527,0.209561,...,SUP,USD,Kraken,39287.474016,1.700000e+03,False,full,2.91,True,0.976
5,125,3.338000e-01,3.351200e-01,4.805600e-01,1.829569,0.056710,52.560000,0.608000,-0.309475,0.667600,...,WFB,USD,Kraken,47.239530,1.200000e+01,False,young,1.43,True,0.969
6,172,4.384000e-02,4.384000e-02,8.135750e-02,2.479959,0.196344,106.104651,0.593023,-0.489877,0.509294,...,BDX,USD,Kraken,0.000000,6.000000e+01,False,full,1.86,True,0.967
7,116,6.240000e-04,5.082500e-04,9.025000e-04,2.211438,0.155854,59.784483,0.603448,-0.572603,0.427397,...,SHAPE,USD,Kraken,5826.648369,7.800000e+03,False,young,1.78,True,0.965
8,220,6.720000e-07,5.956000e-07,9.330500e-07,1.833024,0.005343,89.590909,0.600000,-0.088195,0.622222,...,KIN,USD,Kraken,321.119767,7.000000e+06,False,full,1.57,True,0.964
9,164,6.329000e-03,6.310300e-03,1.191445e-02,1.736829,0.126340,75.670732,0.597561,-0.646088,0.353912,...,INX,EUR,Kraken,2424.538657,6.000000e+02,False,full,1.89,True,0.960


In [4]:
# ── Cell 4: pick the markets that get the full engine ────────────────
# v11: the model-free grid-harvest rank (zig-zag swings net of round-trip cost)
# is blended with the base composite. harvest_* columns are on DAILY bars here
# (breadth); the engine recomputes them on intraday bars for the finalists.
yamls = lr.discover_controller_yamls(YAML_ROOTS, exchange=EXCHANGE)
yamls = lr.discover_controller_yamls(YAML_ROOTS,
                                     pattern='*range_inventory_ladder*.y*ml',
                                     exchange=EXCHANGE)
yaml_markets = [lr._pair_key(p['trading_pair']) for p in yamls]
print(f'controller YAMLs found: {len(yamls)} → {yaml_markets}')

ranked = lr.rank_review_markets(screen_df, hist_d, CFG)
display(ranked.head(25))

col = 'base' if 'base' in ranked.columns else 'pairkey'
# dust filter applies HERE (screened candidates only); YAML/always/focus bypass
eligible = [pk for pk in ranked[col] if VOL_USD.get(pk, 0) >= CFG['min_vol_usd']]
dropped = len(ranked) - len(eligible)
if dropped:
    print(f'dust filter: {dropped} screened markets under ${CFG["min_vol_usd"]:,.0f} 24h volume')
top_screen = eligible[:CFG['rc_top_n_from_screen']]
wanted = lr.merge_unique_markets(top_screen, CFG['always_review_markets'],
                                 CFG['focus_markets'], yaml_markets)
review, missing = lr.resolve_present(uni, wanted)
if missing:
    print('not on this exchange / below volume floor:', missing)
print(f'{len(review)} markets selected for the v11 engine')


controller YAMLs found: 2 → ['DASH/EUR', 'XMR/USD']


,base,days,price,low,high,ann_vol,er,cross_per_yr,in_band,net_return,...,range_x,qualifies,composite,harvest_rt_cost_pct,harvest_1.5x_pct_mo,harvest_2x_pct_mo,harvest_3x_pct_mo,harvest_best_pct_mo,harvest_best_gap_pct,review_rank
0,SUP/USD,158,2.709000e-03,2.506500e-03,7.299800e-03,3.535260,0.089982,62.373418,0.594937,-0.735527,...,2.91,True,0.976,12.500,15.127,24.204,19.363,24.204,25.000,0.995
1,CORN/USD,220,2.401000e-02,2.332000e-02,7.156150e-02,3.089881,0.109080,49.772727,0.600000,-0.713484,...,3.07,True,0.948,11.774,7.577,14.433,18.763,18.763,35.323,0.980
2,SUP/EUR,158,2.719000e-03,2.719000e-03,6.030000e-03,3.929752,0.080275,36.962025,0.715190,-0.682582,...,2.22,True,0.919,12.500,7.261,14.522,19.363,19.363,37.500,0.974
3,ELX/EUR,220,1.313000e-03,8.599500e-04,3.624800e-03,3.363477,0.121512,49.772727,0.609091,-0.815070,...,4.22,True,0.932,12.500,8.427,14.556,18.387,18.387,37.500,0.974
4,UNFI/EUR,220,3.140000e-02,2.149500e-02,9.430000e-02,3.259070,0.046290,36.500000,0.600000,-0.600000,...,4.39,True,0.903,12.500,8.427,12.258,21.452,21.452,37.500,0.973
5,TEER/USD,220,8.110000e-03,8.110000e-03,1.961000e-02,2.472238,0.102744,36.500000,0.772727,-0.654894,...,2.42,True,0.915,11.060,9.830,16.947,18.981,18.981,33.181,0.972
6,SRM/EUR,220,5.770000e-03,3.845500e-03,8.853000e-03,3.144779,0.047798,48.113636,0.600000,-0.480180,...,2.30,True,0.954,12.500,9.577,16.089,13.790,16.089,25.000,0.969
7,BILLY/EUR,220,1.414000e-03,1.314000e-03,2.962600e-03,1.913963,0.052594,66.363636,0.600000,-0.533795,...,2.25,True,0.925,12.105,13.355,15.581,17.806,17.806,36.316,0.969
8,GHIBLI/EUR,220,2.550000e-04,2.498500e-04,7.700500e-04,2.049281,0.038210,77.977273,0.600000,-0.543828,...,3.08,True,0.913,12.500,10.726,14.556,18.387,18.387,37.500,0.969
9,RAILS/EUR,159,5.262000e-02,4.511400e-02,2.468400e-01,3.081792,0.166257,41.320755,0.597484,-0.850977,...,5.47,True,0.887,12.500,13.829,18.038,21.646,21.646,37.500,0.968


dust filter: 940 screened markets under $10,000 24h volume
44 markets selected for the v11 engine


In [5]:
# ── Cell 5: history — hourly Nx6 for search/WF/holdout, 5m Nx6 for reports ─
# v11 bars carry QUOTE VOLUME (column 6) so the sim can cap fills against what
# actually traded. Kraken pairs ride the MEXC proxy (USD→USDT alias, guarded by
# the last-price check); Kraken-native OHLC is a 30d fallback (720-candle cap).
hist_h = lr.prefetch_bars6(uni, CFG, cache, pairs=review, interval='60m',
                           min_days_key='rc_min_hourly_days')
hist_h = lr.with_daily_fallback(hist_h, hist_d, CFG, review)   # daily fallback is flagged
print(pd.Series({pk: v['granularity'] for pk, v in hist_h.items()}).value_counts().to_string())
vol_missing = [pk for pk, v in hist_h.items() if not v.get('vol_known', False)]
if vol_missing:
    print('volume unknown (volume cap disabled) for:', vol_missing)

if CFG['rc_report_interval'] not in ('60m', '1h'):
    hist_report = lr.prefetch_bars6(uni, CFG, cache, pairs=review,
                                    interval=CFG['rc_report_interval'],
                                    min_days_key='rc_min_intraday_days')
    print(pd.Series({pk: v['granularity'] for pk, v in hist_report.items()}).value_counts().to_string())
else:
    hist_report = hist_h
missing_5m = [pk for pk in review if pk in hist_h and pk not in hist_report]
if missing_5m:
    print('report falls back to search-granularity bars for:', missing_5m)


  ...bars6[60m] 10/44
  ...bars6[60m] 20/44
  ...bars6[60m] 30/44
  ...bars6[60m] 40/44
bars6(60m): 25/44 markets in 35s
1h    25
1d    19
volume unknown (volume cap disabled) for: ['SUP/USD', 'REPPO/USD', 'UAI/EUR', 'AI/USD', 'UNFI/USD', 'UAI/USD', 'BMB/USD', 'BASED/USD', 'REPPO/EUR', 'ZAMA/USD', 'VELVET/USD', 'LIT/USD', 'NIL/EUR', 'XPL/EUR', 'BLUAI/EUR', 'HMSTR/EUR', 'PUMP/EUR', 'PACT/USD', 'DASH/EUR']
  ...bars6[5m] 10/44
  ...bars6[5m] 20/44
  ...bars6[5m] 30/44
  ...bars6[5m] 40/44
bars6(5m): 21/44 markets in 34s
5m    21
report falls back to search-granularity bars for: ['SUP/USD', 'SPACE/USD', 'REPPO/USD', 'UAI/EUR', 'AI/USD', 'UNFI/USD', 'UAI/USD', 'BMB/USD', 'BASED/USD', 'REPPO/EUR', 'GWEI/USD', 'ZAMA/USD', 'VELVET/USD', 'LIT/USD', 'NIL/EUR', 'XPL/EUR', 'BLUAI/EUR', 'HMSTR/EUR', 'ACU/USD', 'PUMP/EUR', 'SENT/USD', 'PACT/USD', 'DASH/EUR']


In [6]:
# ── Cell 6: YOUR live controller YAMLs, first-class, on the same engine ──
# Genuinely out-of-sample for any ladder you wrote before this window ends.
# v11 engine: penetration + volume caps + book-spread slip apply here too, so
# these numbers are the closest thing to a live replay in the whole pipeline.
yaml_hist = {pk: hist_report.get(pk, hist_h.get(pk)) for pk in set(list(hist_h) + list(hist_report))}
yaml_hist = {pk: v for pk, v in yaml_hist.items() if v is not None}
# Mark which YAMLs are LIVE (ground truth) vs untraded DRAFTS (hypotheses).
# Leave empty to treat them all as live; otherwise list deployed controller_ids.
DEPLOYED_CONTROLLERS = set()      # e.g. {'range_inventory_ladder_xmr_V1'}
yaml_df, yaml_reports = lr.evaluate_controller_yamls(yamls, yaml_hist, uni, CFG)
if not yaml_df.empty and 'controller' in yaml_df.columns and DEPLOYED_CONTROLLERS:
    yaml_df.insert(1, 'status', yaml_df.controller.map(
        lambda c: 'LIVE' if c in DEPLOYED_CONTROLLERS else 'DRAFT'))
    print('LIVE = deployed (ground truth) | DRAFT = never traded (a backtest of a proposal)')
if not yaml_df.empty:
    display(yaml_df)
    # ---- v11.3: LIVE-STRATEGY HEALTH -- recent form, judged on two axes ----
    # A 180-day headline can hide a strategy that died a month ago. This is
    # the table that makes "no longer profitable" impossible to miss.
    health = lr.live_strategy_health(yaml_df, yaml_reports, CFG)
    print()
    lr.print_health_banners(health)
    display(health)
    health.to_csv(f'{PREFIX}_live_health.csv', index=False)
    for cid, rep in yaml_reports.items():
        print(f"\n=== {cid} — passed={rep['passed']}"
              f"  {('; '.join(rep['failed_gates']) if rep['failed_gates'] else '')} ===")
        display(rep['blocks'])
else:
    print('no controller YAMLs matched this exchange — set YAML_ROOTS in Cell 1')


,controller,pair,src,days,blocks,pnl_pct,hold_pct,edge_pct,edge_pos_rate,abs_pos_rate,worst_block_edge,trades,trades_per_month,med_trades_per_block,maxdd,endinv,stress_edge_pct,vol_capped_fills,passed,failed_gates
0,k_range_inventory_ladder_dash_eur_V1,DASH/EUR,Kraken,180.0,12,69.474,0.000,69.474,0.778,0.636,-2.418,80,13.5,7.0,15.795,45.5,39.331,0,False,median 7 trades/ACTIVE block < 8; clean: media...
1,k_range_inventory_ladder_xmr_usd_V1,XMR/USD,MEXC(XMRUSDT),180.0,12,-8.182,-40.869,32.687,0.727,0.778,-7.079,498,84.1,63.0,53.158,97.9,25.041,0,True,


,controller,pair,status,recent_blocks,recent_days,recent_pnl_pct,recent_hold_pct,recent_edge_pct,recent_trades,recent_two_sided_blocks,last_block_pnl_pct,full_pnl_pct,full_edge_pct
0,k_range_inventory_ladder_dash_eur_V1,DASH/EUR,HEALTHY,3,45.0,15.50,0.0,15.50,23,3,1.799,69.474,69.474
1,k_range_inventory_ladder_xmr_usd_V1,XMR/USD,HEALTHY,3,45.0,16.43,-5.7,22.13,177,3,7.928,-8.182,32.687



=== k_range_inventory_ladder_dash_eur_V1 — passed=False  median 7 trades/ACTIVE block < 8; clean: median 7 trades/ACTIVE block < 8 ===


,block,start,end,days,pnl_pct,hold_pct,edge_pct,buy_fills,sell_fills,trades,two_sided,in_band_pct,in_fit,partial
0,1,2026-01-13,2026-01-27,15.0,0.000,0.0,0.000,0,0,0,False,0.000,False,False
1,2,2026-01-28,2026-02-11,15.0,5.429,0.0,5.429,8,3,11,True,0.800,False,False
2,3,2026-02-12,2026-02-26,15.0,7.202,0.0,7.202,7,4,11,True,1.000,False,False
3,4,2026-02-27,2026-03-13,15.0,-2.079,0.0,-2.079,5,1,6,True,0.933,False,False
4,5,2026-03-14,2026-03-28,15.0,-0.220,0.0,-0.220,8,2,10,True,1.000,False,False
5,6,2026-03-29,2026-04-12,15.0,29.359,0.0,29.359,1,6,7,True,0.667,False,False
6,7,2026-04-13,2026-04-27,15.0,-2.418,0.0,-2.418,1,3,4,True,1.000,False,False
7,8,2026-04-28,2026-05-12,15.0,4.958,0.0,4.958,0,7,7,False,0.667,False,False
8,9,2026-05-13,2026-05-27,15.0,-0.129,0.0,-0.129,0,0,0,False,0.867,False,False
9,10,2026-05-28,2026-06-11,15.0,12.926,0.0,12.926,5,4,9,True,1.000,False,False



=== k_range_inventory_ladder_xmr_usd_V1 — passed=True   ===


,block,start,end,days,pnl_pct,hold_pct,edge_pct,buy_fills,sell_fills,trades,two_sided,in_band_pct,in_fit,partial
0,1,2026-01-13,2026-01-28,15.0,-24.055,-24.055,0.000,0,0,0,False,0.000,False,False
1,2,2026-01-28,2026-02-12,15.0,-12.309,-22.378,10.069,60,47,107,True,0.540,False,False
2,3,2026-02-12,2026-02-27,15.0,10.242,2.433,7.809,42,54,96,True,0.999,False,False
3,4,2026-02-27,2026-03-14,15.0,0.340,3.467,-3.126,2,20,22,True,1.000,False,False
4,5,2026-03-14,2026-03-29,15.0,0.605,-6.171,6.776,17,5,22,True,1.000,False,False
5,6,2026-03-29,2026-04-13,15.0,6.358,3.921,2.438,26,42,68,True,1.000,False,False
6,7,2026-04-13,2026-04-28,15.0,0.139,7.219,-7.079,0,6,6,False,0.818,False,False
7,8,2026-04-28,2026-05-13,15.0,0.079,4.015,-3.936,0,0,0,False,0.134,False,False
8,9,2026-05-13,2026-05-28,15.0,-0.156,-7.644,7.488,0,0,0,False,0.121,False,False
9,10,2026-05-28,2026-06-12,15.0,10.634,-0.242,10.876,26,45,71,True,0.825,False,False


In [7]:
# ── Cell 7: full engine per market (WF + HOLDOUT fit + frozen block report) ──
evals = {}
t0 = time.time()
for i, pk in enumerate(review, 1):
    if pk not in hist_h:
        print(f'[{i}/{len(review)}] {pk}: no usable history'); continue
    try:
        ev = lr.evaluate_market(pk, hist_h, uni, CFG, hist_report=hist_report)
    except Exception as e:
        print(f'[{i}/{len(review)}] {pk}: ERROR {e}'); continue
    evals[pk] = ev
    s, wf, ho = ev['report']['summary'], ev['wf'], ev.get('holdout') or {}
    print(f"[{i}/{len(review)}] {pk:14s} rep={ev['granularity']}/fit={ev['search_granularity']} "
          f"edge={s['edge_pct']:+7.2f}% epr={s.get('edge_pos_rate')} "
          f"clean_epr={s.get('clean_edge_pos_rate')} "
          f"HOLDOUT={ho.get('edge_pct', float('nan')):+.2f}% "
          f"blocks_pass={ev['report']['passed']} wf_pass={wf.get('wf_pass')}")
print(f'engine done in {time.time()-t0:.0f}s')


[1/44] SUP/USD        rep=1d/fit=1d edge= +77.28% epr=0.5 clean_epr=0.0 HOLDOUT=+29.16% blocks_pass=False wf_pass=False
[2/44] SPACE/USD      rep=1h/fit=1h edge=+108.63% epr=0.909 clean_epr=1.0 HOLDOUT=+0.82% blocks_pass=True wf_pass=False
[3/44] REPPO/USD      rep=1d/fit=1d edge=+166.84% epr=0.8 clean_epr=None HOLDOUT=-11.71% blocks_pass=False wf_pass=False
[4/44] UAI/EUR        rep=1d/fit=1d edge=+232.05% epr=0.9 clean_epr=1.0 HOLDOUT=-5.44% blocks_pass=False wf_pass=False
[5/44] AI/USD         rep=1d/fit=1d edge= +32.96% epr=0.75 clean_epr=None HOLDOUT=+12.49% blocks_pass=False wf_pass=False
[6/44] APR/USD        rep=5m/fit=1h edge=+162.03% epr=0.75 clean_epr=0.667 HOLDOUT=+6.18% blocks_pass=False wf_pass=False
[7/44] UNFI/USD       rep=1d/fit=1d edge= +48.94% epr=0.667 clean_epr=0.0 HOLDOUT=+15.37% blocks_pass=False wf_pass=False
[8/44] UAI/USD        rep=1d/fit=1d edge=  -6.17% epr=0.417 clean_epr=0.333 HOLDOUT=-7.19% blocks_pass=False wf_pass=False
[9/44] BMB/USD        rep=1d/fi

In [8]:
# ── Cell 8: rolling walk-forward summary (secondary, leakage-safe) ────
wf_rows = pd.DataFrame([dict(market=pk, **{k: v for k, v in ev['wf'].items() if k != 'folds'})
                        for pk, ev in evals.items() if ev])
if not wf_rows.empty:
    wf_rows = wf_rows.sort_values(['wf_pass', 'edge_pos_rate', 'median_edge_pct'],
                                  ascending=[False, False, False]).reset_index(drop=True)
display(wf_rows)


,market,n_folds,edge_pos_rate,two_sided_rate,median_edge_pct,median_test_pct,median_stress_edge,worst_edge_pct,median_trades,total_trades,wf_pass,note
0,XMR/USDT,8,0.625,0.625,1.494,1.160,0.948,-4.852,20.0,214.0,True,NaN
1,B2/USD,8,0.625,0.875,1.067,2.022,0.755,-4.022,21.5,210.0,True,NaN
2,XMR/USD,8,0.625,0.500,0.376,1.247,-0.114,-4.962,21.0,199.0,True,NaN
3,PUMP/EUR,8,0.625,0.500,0.292,-0.554,-0.095,-7.267,10.5,89.0,True,NaN
4,REPPO/EUR,1,1.000,1.000,21.083,22.119,-1.940,21.083,14.0,14.0,False,NaN
5,Q/USD,8,0.750,0.750,6.991,10.341,2.205,-57.655,42.0,355.0,False,NaN
6,XPL/USD,8,0.750,0.750,3.957,9.425,2.543,-10.361,38.0,317.0,False,NaN
7,APR/USD,8,0.750,0.875,1.900,6.257,0.472,-10.121,43.5,411.0,False,NaN
8,AKE/USD,8,0.625,0.875,4.532,3.837,1.937,-74.472,33.0,313.0,False,NaN
9,STABLE/USD,8,0.625,0.875,4.202,6.376,2.350,-8.188,41.5,305.0,False,NaN


In [9]:
# ── Cell 9: finalize (v10 gates + holdout/clean gates + sizing), write artifacts ─
final_df, configs = lr.finalize_v11(evals, uni, CFG, yaml_reports)
display(final_df)

print('\n— HOLDOUT (the deciding out-of-sample evidence) —')
ho_view = final_df[['base', 'validation', 'holdout_edge_pct', 'holdout_trades',
                    'holdout_two_sided', 'holdout_active', 'edge_pct',
                    'clean_edge_pos_rate', 'harvest_best_pct_mo',
                    'max_fund', 'max_fund_quote', 'depth_2pct', 'gates']]
display(ho_view)

block_tables = {pk: ev['report']['blocks'] for pk, ev in evals.items() if ev}
written = lr.save_v11_outputs(PREFIX, final_df, configs, wf_rows, yaml_df, block_tables,
                              metadata=dict(exchange=EXCHANGE, run=RUN_TS,
                                            cfg={k: str(v) for k, v in CFG.items() if str(k).startswith('rc_')},
                                            yaml_paths=[p['path'] for p in yamls]))
# ---- v11.3: FULLY DEPLOYABLE YAMLs ----------------------------------------
# One ready-to-run controller config per market that is CONFIRMED, a strong
# CANDIDATE (GATED on the WF process check only, with a two-sided holdout and
# clean blocks), or ALREADY LIVE (a REFRESH re-fit at today's anchor -- never
# drop one over a running config without comparing rungs; bump the id and
# start a clean state file if you adopt it).
deployed_pairs = [lr._pair_key(p['trading_pair']) for p in yamls
                  if not DEPLOYED_CONTROLLERS
                  or p.get('controller_id', p.get('id')) in DEPLOYED_CONTROLLERS]
yaml_paths = lr.save_deploy_yamls(ART / 'deploy_yamls', configs, final_df, CFG,
                                  deployed_pairs=deployed_pairs, run_id=RUN_TS[:8])
print(f'\n{len(yaml_paths)-1} deployable YAMLs -> {ART}/deploy_yamls (see INDEX.md)')
print('Read:', f'{PREFIX}_copy_paste_ladders.md')


,base,trading_pair,src,granularity,validation,rungs,family,spacing,weights,pnl_pct,...,quote_usd_rate,max_fund_quote,max_fund,spread_pct,depth_2pct,depth_used,depth_basis,depth_ladder_band,book_total_usd,gates
0,XMR/USD,XMR-USD,MEXC(XMRUSDT),5m,CONFIRMED,10+7,quantile,back_loaded,slight_deep,21.576,...,1.00000,10000.000,10000,0.0247,1066574.0,2749939.0,ladder_band,2749939.0,3.613548e+06,
1,XMR/USDT,XMR-USDT,MEXC,5m,CONFIRMED,8+8,quantile,back_loaded,slight_deep,19.475,...,1.00000,10000.000,10000,0.0586,1086526.0,2379701.0,ladder_band,2379701.0,1.304605e+09,
2,SPACE/USD,SPACE-USD,MEXC(SPACEUSDT),1h,GATED,8+7,quantile,back_loaded,slight_deep,89.220,...,1.00000,400.000,400,0.0991,16858.0,22113.0,ladder_band,22113.0,2.926000e+05,rolling WF (process check) failed
3,STABLE/USD,STABLE-USD,MEXC(STABLEUSDT),5m,GATED,10+8,quantile,back_loaded,slight_deep,350.651,...,1.00000,300.000,300,0.0275,42991.0,72828.0,ladder_band,72828.0,1.612890e+05,rolling WF (process check) failed
4,XPL/USD,XPL-USD,MEXC(XPLUSDT),5m,GATED,10+9,quantile,linear,slight_deep,84.038,...,1.00000,3500.000,3500,0.1134,192446.0,275023.0,ladder_band,275023.0,2.096382e+06,rolling WF (process check) failed
5,DASH/USD,DASH-USD,MEXC(DASHUSDT),5m,GATED,9+8,quantile,back_loaded,slight_deep,57.831,...,1.00000,2750.000,2750,0.0725,197834.0,669286.0,ladder_band,669286.0,1.787337e+06,rolling WF (process check) failed
6,REPPO/EUR,REPPO-EUR,Kraken,1d,SUSPECT,9+10,quantile,back_loaded,slight_deep,288.111,...,1.13935,175.539,200,4.1724,0.0,3990.0,ladder_band,3990.0,1.430900e+04,only 5 full blocks (< 6); clean: only 1 out-of...
7,PTB/USD,PTB-USD,MEXC(PTBUSDT),5m,SUSPECT,7+5,quantile,back_loaded,slight_deep,-27.190,...,1.00000,400.000,400,0.6205,3945.0,10644.0,ladder_band,10644.0,2.014557e+06,only 3 ACTIVE blocks (< 4) -- band rarely visi...
8,REPPO/USD,REPPO-USD,Kraken,1d,SUSPECT,9+10,volatility,back_loaded,slight_deep,220.661,...,1.00000,1250.000,1250,2.2414,1711.0,6253.0,ladder_band,6253.0,4.105370e+05,only 5 full blocks (< 6); clean: only 1 out-of...
9,PACT/USD,PACT-USD,Kraken,1d,SUSPECT,8+4,volatility,back_loaded,slight_deep,59.237,...,1.00000,500.000,500,4.2760,0.0,1141.0,min_band_5pct,334.0,5.291380e+05,worst block edge -13.91% < -12.5%; median 6 tr...



— HOLDOUT (the deciding out-of-sample evidence) —


,base,validation,holdout_edge_pct,holdout_trades,holdout_two_sided,holdout_active,edge_pct,clean_edge_pos_rate,harvest_best_pct_mo,max_fund,max_fund_quote,depth_2pct,gates
0,XMR/USD,CONFIRMED,3.791,43,True,True,46.822,0.833,32.627,10000,10000.000,1066574.0,
1,XMR/USDT,CONFIRMED,3.498,24,True,True,44.721,0.800,32.627,10000,10000.000,1086526.0,
2,SPACE/USD,GATED,0.817,24,False,True,108.630,1.000,70.789,400,400.000,16858.0,rolling WF (process check) failed
3,STABLE/USD,GATED,6.907,38,True,True,276.609,0.667,54.224,300,300.000,42991.0,rolling WF (process check) failed
4,XPL/USD,GATED,16.274,61,True,True,105.057,0.667,51.664,3500,3500.000,192446.0,rolling WF (process check) failed
5,DASH/USD,GATED,3.542,19,True,True,76.012,0.800,44.138,2750,2750.000,197834.0,rolling WF (process check) failed
6,REPPO/EUR,SUSPECT,-11.454,15,False,True,232.837,NaN,20.813,200,175.539,0.0,only 5 full blocks (< 6); clean: only 1 out-of...
7,PTB/USD,SUSPECT,9.385,27,True,True,13.861,1.000,70.579,400,400.000,3945.0,only 3 ACTIVE blocks (< 4) -- band rarely visi...
8,REPPO/USD,SUSPECT,-11.712,17,False,True,166.838,NaN,39.304,1250,1250.000,1711.0,only 5 full blocks (< 6); clean: only 1 out-of...
9,PACT/USD,SUSPECT,-37.351,3,False,True,95.950,0.500,13.582,500,500.000,0.0,worst block edge -13.91% < -12.5%; median 6 tr...


  wrote artifacts/kraken/20260713-040411/files/KRAKEN_recycle_v9_final_summary.csv
  wrote artifacts/kraken/20260713-040411/files/KRAKEN_recycle_v9_walkforward_summary.csv
  wrote artifacts/kraken/20260713-040411/files/KRAKEN_recycle_v9_live_yaml_summary.csv
  wrote artifacts/kraken/20260713-040411/files/KRAKEN_recycle_v9_block_details.csv
  wrote artifacts/kraken/20260713-040411/files/KRAKEN_recycle_v9_holdout_summary.csv
  wrote artifacts/kraken/20260713-040411/files/KRAKEN_recycle_v9_deploy_config.json
  wrote artifacts/kraken/20260713-040411/files/KRAKEN_recycle_v9_copy_paste_ladders.md
  wrote deployable YAML artifacts/kraken/20260713-040411/files/deploy_yamls/k_range_inventory_ladder_dash_eur_auto_20260713.yml
  wrote deployable YAML artifacts/kraken/20260713-040411/files/deploy_yamls/k_range_inventory_ladder_dash_usd_auto_20260713.yml
  wrote deployable YAML artifacts/kraken/20260713-040411/files/deploy_yamls/k_range_inventory_ladder_stable_usd_auto_20260713.yml
  wrote deployab

In [10]:
# ── Cell 10: VERIFY THE BOOKS for the shortlist ───────────────────────
# v11.2: reports the full DEPTH PROFILE (+-2/5/10/25% + whole book) and the
# depth inside each market's OWN deployed ladder span. A fixed +-2% reading
# misjudges ladder-shaped books: NonKYC DASH/USDT quotes dust at the touch
# and parks real size ~2.6% out (+-2% = $2, +-5% = $29,820, book = $33,509).
short = final_df[((final_df.holdout_edge_pct > 1) &
                  (final_df.clean_edge_pos_rate >= 0.6)) |
                 (final_df.validation != 'SUSPECT')]
short_pairs = list(short.base.head(12))
print('verifying books for:', short_pairs)
lads = {pk: evals[pk]['deployed'] for pk in short_pairs if pk in evals}
book_df = lr.verify_books(uni, short_pairs, CFG, samples=5, pause=4.0, ladders=lads)
display(book_df)
book_df.to_csv(f'{PREFIX}_book_verification.csv', index=False)
print('READ THE PROFILE, not one band:')
print('  depth_2pct tiny but depth_5pct large -> dust quoted at the touch, real size')
print('     parked further out. NOT a thin market -- depth_ladder_band is the truth.')
print('  depth_ladder_band = liquidity inside YOUR rungs (what can actually fill them).')
print('  flaky=True -> book swinging >5x across samples; trust no single reading.')
print('  thin_med=True with steady samples -> genuinely thin: size down or skip.')


verifying books for: ['XMR/USD', 'XMR/USDT', 'SPACE/USD', 'STABLE/USD', 'XPL/USD', 'DASH/USD', 'PTB/USD', 'APR/USD', 'B2/USD', 'NIL/USD', 'PLAY/USD', 'BMB/USD']
  book check XMR/USD: {'market': 'XMR/USD', 'samples_ok': 5, 'spread_pct': 0.0308, 'book_total_usd': 3614883.0, 'n_bids': 500.0, 'n_asks': 500.0, 'depth_2pct': 1069385.0, 'depth_5pct': 2288222.0, 'depth_10pct': 2501917.0, 'depth_25pct': 3097487.0, 'flaky': False, 'depth_ladder_band': 2751082.0, 'thin_med': False, 'size_suggestion': 10000.0}
  book check XMR/USDT: {'market': 'XMR/USDT', 'samples_ok': 5, 'spread_pct': 0.0801, 'book_total_usd': 1304606727.0, 'n_bids': 170.0, 'n_asks': 408.0, 'depth_2pct': 1088572.0, 'depth_5pct': 2129727.0, 'depth_10pct': 2259582.0, 'depth_25pct': 2462155.0, 'flaky': False, 'depth_ladder_band': 2373658.0, 'thin_med': False, 'size_suggestion': 10000.0}
  book check SPACE/USD: {'market': 'SPACE/USD', 'samples_ok': 5, 'spread_pct': 0.0661, 'book_total_usd': 292775.0, 'n_bids': 74.0, 'n_asks': 87.0, '

,market,samples_ok,spread_pct,book_total_usd,n_bids,n_asks,depth_2pct,depth_5pct,depth_10pct,depth_25pct,flaky,depth_ladder_band,thin_med,size_suggestion
0,XMR/USD,5,0.0308,3.614883e+06,500.0,500.0,1069385.0,2288222.0,2501917.0,3097487.0,False,2751082.0,False,10000.0
1,XMR/USDT,5,0.0801,1.304607e+09,170.0,408.0,1088572.0,2129727.0,2259582.0,2462155.0,False,2373658.0,False,10000.0
2,SPACE/USD,5,0.0661,2.927750e+05,74.0,87.0,16876.0,19828.0,24602.0,27278.0,False,22083.0,False,10000.0
3,STABLE/USD,5,0.0275,1.604800e+05,67.0,46.0,42242.0,56990.0,59811.0,114478.0,False,72019.0,False,10000.0
4,XPL/USD,5,0.1137,2.097971e+06,98.0,311.0,194249.0,228215.0,245310.0,288289.0,False,276915.0,False,10000.0
5,DASH/USD,5,0.0811,2.041993e+06,428.0,500.0,255364.0,377708.0,442730.0,671865.0,False,663726.0,False,10000.0
6,PTB/USD,5,0.4167,2.015493e+06,44.0,177.0,4415.0,9660.0,13223.0,15168.0,False,9227.0,False,4614.0
7,APR/USD,5,0.3901,5.671900e+04,43.0,33.0,14783.0,18669.0,29833.0,33502.0,False,6087.0,False,3044.0
8,B2/USD,5,0.2603,6.377400e+04,28.0,37.0,5720.0,6298.0,23390.0,24052.0,False,24052.0,False,10000.0
9,NIL/USD,5,0.2861,5.752430e+05,30.0,68.0,55976.0,73017.0,74353.0,83371.0,False,81239.0,False,10000.0


READ THE PROFILE, not one band:
  depth_2pct tiny but depth_5pct large -> dust quoted at the touch, real size
     parked further out. NOT a thin market -- depth_ladder_band is the truth.
  depth_ladder_band = liquidity inside YOUR rungs (what can actually fill them).
  flaky=True -> book swinging >5x across samples; trust no single reading.
  thin_med=True with steady samples -> genuinely thin: size down or skip.


In [11]:
# ── Cell 11 (optional): calibrate the fill model against LIVE fills ───
# Point each pair at ONE OR MANY controller diagnostic JSONL files -- a single
# path, a list, and/or glob patterns. Rotated/overlapping logs are safe: fills
# are merged chronologically and deduplicated on (ts, side, price, amount).
#
# How much data is meaningful (rule of thumb: count error ~ 1/sqrt(N fills)):
#   < 15 fills / <10d  -> directional hint only, do NOT tune knobs
#   30+ fills, 15d+    -> first honest sim/live ratio (one full block)
#   60+ fills, 30d+    -> tune rc_volume_cap_frac / rc_fill_penetration_pct
#                         until sim_over_live_fill_ratio ~= 0.8-1.2
# If your controllers have been logging since deployment, you may already
# have months of data -- point globs at the whole log directory.
DIAG_FILES = {
    # 'XMR/USDT': '/path/to/xmr_diag.jsonl',                       # single file
    # 'XMR/USDT': ['/logs/xmr_*.jsonl', '/old_logs/xmr_2026*.jsonl'],  # globs+lists
}
DIAG_FILES = {
    'XMR/USD': ['./diagnostics/kraken/range_inventory_ladder_xmr_usd_diagnostic*.jsonl'],  # globs+lists
}

for pk, paths in DIAG_FILES.items():
    print(f'=== {pk} ===')
    fills = lr.collect_jsonl_fills(paths)
    print(lr.fills_sufficiency(fills))
    if fills.empty:
        first = lr.first_existing(paths)          # glob-safe (v11.1.2 fix)
        if first is None:
            print(f'no files matched: {paths} -- check the path/glob')
        else:
            print(f'no fills extracted from {first} -- schema sniff:')
            display(lr.summarize_diagnostic_jsonl(first))
        continue
    print(fills.groupby('source_file').size().to_string())
    ladder = next((lr.controller_to_ladder(p, CFG) for p in yamls
                   if lr._pair_key(p['trading_pair']) == pk), None)
    h = hist_report.get(pk, hist_h.get(pk))
    if ladder is not None and h is not None:
        pdec = uni.get('pdec', {}).get(pk)
        book_half = lr.market_book_half_spread_pct(uni, pk, CFG)
        cfg_m, _ = lr.quote_scaled_cfg(uni, pk, CFG)
        cmp_ = lr.compare_live_vs_sim_v11(fills, h['bars'], ladder, cfg_m,
                                          pdec=pdec, book_half=book_half)
        for k, v in cmp_.items():
            print(f'  {k:28s} {v}')
    else:
        print('  (no matching controller YAML or history for the sim side)')

# refresh the health table with live-fill recency, now that fills are loaded
if DIAG_FILES and yaml_reports:
    fills_map = {pk: lr.collect_jsonl_fills(paths) for pk, paths in DIAG_FILES.items()}
    health = lr.live_strategy_health(yaml_df, yaml_reports, CFG, fills_by_pair=fills_map)
    lr.print_health_banners(health)
    display(health)
    health.to_csv(f'{PREFIX}_live_health.csv', index=False)


=== XMR/USD ===
1 fill(s), window ~0.0h -> TOO THIN: a rate needs at least a few fills spread over time; keep logging
source_file
./diagnostics/kraken/range_inventory_ladder_xmr_usd_diagnostic_20260709-1747-20260709-174713.jsonl    1
  note                         only 1 fill(s) over 0.0h -- no meaningful rate; keep logging
  fills_sufficiency            1 fill(s), window ~0.0h -> TOO THIN: a rate needs at least a few fills spread over time; keep logging


,controller,pair,status,recent_blocks,recent_days,recent_pnl_pct,recent_hold_pct,recent_edge_pct,recent_trades,recent_two_sided_blocks,last_block_pnl_pct,full_pnl_pct,full_edge_pct,days_since_live_fill
0,k_range_inventory_ladder_dash_eur_V1,DASH/EUR,HEALTHY,3,45.0,15.50,0.0,15.50,23,3,1.799,69.474,69.474,NaN
1,k_range_inventory_ladder_xmr_usd_V1,XMR/USD,HEALTHY,3,45.0,16.43,-5.7,22.13,177,3,7.928,-8.182,32.687,0.6


## Reading the results (v11 evidence hierarchy)

Work down this list; each level is weaker evidence than the one above it.

1. **`holdout_edge_pct`** (Cell 9 holdout view / `*_holdout_summary.csv`) — the fitted,
   train-anchored ladder on 15 days it never saw. This is the only fully out-of-sample
   number for the geometry you deploy. Positive and two-sided ⇒ real signal; dormant
   (`holdout_active=False`) ⇒ no evidence either way, and the gate says so.
2. **Clean blocks** (`clean_edge_pos_rate`, `clean_worst_block_edge`) — the frozen report's
   blocks that do *not* overlap the fit window. The headline `edge_pct`/`edge_pos_rate`
   still include the fit window (kept for v10 comparability) — prefer the clean ones.
3. **Your live YAMLs** (Cell 6) — ground truth for the fill model itself. If the sim's
   trades/block is far from your JSONL fills, recalibrate in Cell 11 before trusting anything.
4. **Rolling WF** — validates the *fitting process*, not the deployed ladder. A wf_pass with
   a bad holdout means the process is fine but this particular fit is a bad draw: refit later.
5. **`harvest_best_pct_mo`** — model-free ceiling on what any grid can extract at cost.
   If it's near zero, no amount of ladder tuning will make the pair work.

`vol_capped_fills` > 0 tells you the book, not the price path, limited the sim — size DOWN
(`max_fund_value_quote`) rather than dismissing the pair. `fit_score_gap` large (≫ typical)
suggests the winner is a lucky draw among 240 candidates — prefer pairs where holdout,
clean blocks *and* WF agree.

**Kraken note:** most USD pairs are evaluated on MEXC USDT-proxy candles (src `MEXC(XXXUSDT)`) — the last-price guard rejects the proxy when it diverges >5% from Kraken's own ticker. Pairs stuck on `Kraken(720cap)` hourly have only ~30d of history: their reports are short and the holdout may be skipped — treat those as screening-only results.

**Deployment discipline:** deploy only `CONFIRMED`, at the suggested `max_fund_value_quote`
or less, and treat the FIRST live 15-day block as the final gate — if it underperforms the
holdout badly, stop and recalibrate (Cell 11) instead of re-fitting harder.
